# More Than a Hack 2027 · Faz 1: Topa sahip olma (possession)

Bu notebook, verilen tracking verisinden maçın **her karesi** için `H` / `A` / `D` etiketi üretir,
repo'daki resmi metrik koduyla puanlar ve gönderilecek CSV'yi yazar.

**Yaklaşım (baseline'dan farkı):**

| Adım | Baseline (`starter/baseline.py`) | Bu notebook |
|---|---|---|
| Topa kim yakın? | Her karede en yakın oyuncu | Aynı, ama sadece **kontrol** sayılan temaslar: oyuncu ≤ 2 m ve rakipten belirgin şekilde daha yakın, en az ~0,25 sn boyunca |
| Top kaybolunca | `D` deniyor (yanlış!) | Top görünmese de **son kontrol eden takım** topu elinde tutar (ground truth kuralı) |
| Ölü top (`D`) | Sadece "top görünmüyor" | Top saha dışında, top yerde duruyor (faul), Smart Tagging duran topları + bu maçtan **kendi kendine öğrenen** oyuncu-hareketi modeli |
| Duran toptan sonra | Yok | Smart Tagging'deki `Team` etiketi topu kimin başlattığını söyler |
| Titreme | Pencerede çoğunluk oylaması | Değişiklik ancak rakip topu **kontrol edince** (≥ 1 sn tutar ya da takımı bir sonraki dokunuşu da yapar); 1 sn'den kısa spell'ler temizleniyor |
| Top kayıpken el değiştirme | — | Değişim anı, topun görünmediği boşluğun ortasına konuyor |

**Kurallar (önemli):**
- Veriyi (video, tracking, XML, çıktıların satırları) **hiçbir bulut LLM'ine yükleme**. Bana sadece skor tablolarını ve hata mesajlarını gönder.
- `data/` klasörü GitHub'a gitmesin (repo'nun `.gitignore`'u zaten engelliyor).
- Public 5 dakika **parametre ayarlamak için değil**, sadece "pipeline doğru mu" kontrolü için.

**Nasıl çalıştırılır:** Bu dosyayı repo'nun kök klasörüne (yani `games.py`'nin yanına) koy,
verileri `data/tactical/` içine koy ve hücreleri sırayla çalıştır.

In [ ]:
import os
import sys
import time
import difflib
import subprocess
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

ROOT = os.path.abspath(os.getcwd())
assert os.path.exists(os.path.join(ROOT, "games.py")), \
    "Open this notebook from the repository root (the folder that contains games.py)."
for sub in ("", "starter", "evaluation"):
    path = os.path.join(ROOT, sub)
    if path not in sys.path:
        sys.path.insert(0, path)

import games                              # noqa: E402  per-match constants and file paths
import possession_metrics as pm           # noqa: E402  the official metric library
import load_smart_tagging                 # noqa: E402
import load_atd                           # noqa: E402
from baseline import team_of_each_track, team_names  # noqa: E402

GAME = games.GAME
FPS = games.GAMES[GAME]["fps"]                       # 25
PITCH = games.GAMES[GAME]["pitch"]                   # (105.3, 68.0) metres
N_FRAMES = games.source_frames("tactical", GAME)     # 199319
# The two halves as published by the organisers in games.py, as half-open [start, stop) video frames.
HALF_BLOCKS = [(a, b + 1) for _, a, b in games.half_video_blocks(GAME)]

TEAM_NAME = "GolDisi"   # registration formundaki takım adı

print("data dir :", games.DATA_DIR)
print("frames   :", N_FRAMES, "at", FPS, "fps, pitch", PITCH)

## 0. Parametreler

Hepsi futbol mantığından seçildi, public 5 dakikaya göre **ayarlanmadı**. Teknik dokümanda
bu tabloyu olduğu gibi kullanabilirsin.

In [ ]:
PARAMS = dict(
    # ball
    BALL_INTERP_MAX=10,        # frames; linearly bridge ball gaps up to 0.4 s
    BALL_MAX_SPEED_CTRL=12.0,  # m/s; a ball moving faster than this is in flight, not controlled
    OUT_MARGIN=1.0,            # m beyond a line before the ball counts as out of play
    # control
    R_CTRL=2.0,                # m; a player this close to the ball can be controlling it
    MARGIN=0.75,               # m; and must be this much closer than the nearest opponent
    K_CTRL=6,                  # frames (0.24 s); shorter contact = a touch, not control
    K_GAP=3,                   # frames of tolerated dropout inside one control run
    # possession changes
    REQUIRE_CONFIRM=True,      # an opponent's touch changes possession only if it is control:
    KEEP=FPS,                  #   he keeps the ball >= 1 s, or his team also makes the next touch
    GAP_MIDPOINT=True,         # change hidden by a ball gap -> place it mid-gap, not at reappearance
    # dead ball
    MIN_PLAYERS=12,            # fewer tracked players than this = no play (halftime, cuts)
    DEAD_MODEL="learned",      # "learned": self-supervised stoppage model | "otsu": v3 slow-play rule
    MODEL_SMOOTH=91,           # frames; median filter on the stoppage probability (3.6 s)
    MODEL_MIN_DEAD=3 * FPS,    # frames; shortest stoppage the movement model may report
    PLAUSIBLE_DEAD=(0.20, 0.65),  # dead share of the halves; published matches span ~0.30-0.53
    MIN_MODEL_ACC=0.70,        # model must predict one half's pseudo-labels from the other this well
    USE_SLOW_RULE=True,        # (otsu mode) slow players + quiet/invisible ball = stoppage
    SPEED_WIN=2 * FPS,         # frames; window for the mean player speed
    SLOW_MIN=0.8, SLOW_MAX=2.2,  # m/s; clamp for the automatic slow-play threshold
    OTSU_IN_HALVES=True,       # learn the threshold from the two halves only, not warm-up/halftime
    BALL_QUIET_SPEED=2.0,      # m/s; ball this slow (or invisible) is consistent with a stoppage
    RESTART_SPEED=5.0,         # m/s; ball kicked or thrown = restart
    MAX_OUT_HOLD=60 * FPS,     # frames; longest a ball-out stoppage is held without a restart
    USE_STATIC_BALL=True,      # ball lying still with nobody on it = placed for a restart (fouls)
    STATIC_WIN=2 * FPS,        # frames the ball must stay still
    STATIC_SPAN=0.75,          # m; "still" = moves less than this within the window
    STATIC_CLEAR=1.0,          # m; and no player closer than this (a player on the ball = alive)
    MIN_DEAD=3 * FPS,          # frames; shorter dead stretches are dropped
    MAX_ALIVE_GAP_IN_DEAD=int(1.5 * FPS),  # frames; alive blips inside a stoppage are filled
    # smart tagging
    USE_SET_PIECES=True,
    SET_PIECE_LEAD=3.5,        # s; tags open ~3.5 s before the real restart (01_smart_tagging.ipynb)
    RESTART_TOL=3 * FPS,       # frames; a restart this close to an alive stretch owns its start
    USE_PHASE_PRIOR=True,      # use phase tags only where tracking gives no control at all
    PRIOR_MIN_AGREEMENT=0.8,   # a phase code is used only if it agrees (or disagrees) this clearly
    # cleanup
    MIN_SPELL=FPS,             # frames; possessions shorter than 1 s are merged into neighbours
)

## 1. Veriyi yükle

Ham ATD satır formatı: `<frame_id>:<oyuncu>;<oyuncu>;...:<top>`, her grup `x,y,hız`.
Koordinatlar 0–1 arası; metreye çevirmek için saha ölçüsüyle çarpıyoruz.
`frame_id` 1'den başlar, submission 0'dan → `video_frame = frame_id - 1`.

İlk çalıştırma 1–3 dakika sürer; sonuç `data/cache/` içine kaydedilir (`data/` git'e girmez).

In [ ]:
def _parse_players(text, n_slots):
    """(n_slots, 3) float32 array of x, y, speed from one raw ATD player field."""
    flat = text.replace(";", ",").split(",")
    if len(flat) == 3 * n_slots:
        try:
            return np.array(flat, dtype=np.float32).reshape(n_slots, 3)
        except ValueError:
            pass
    out = np.full((n_slots, 3), np.nan, np.float32)
    for i, group in enumerate(text.split(";")[:n_slots]):
        for k, value in enumerate(group.split(",")[:3]):
            try:
                out[i, k] = float(value)
            except ValueError:
                pass
    return out


def load_atd_arrays(use_cache=True):
    """Tracking as arrays in metres, indexed by video frame (= submission row)."""
    meta_path, raw_path = games.require(["atd_meta", "atd_raw"], GAME)
    team = team_of_each_track(meta_path).astype(np.int8)    # per slot: 0 home, 1 away
    n_slots = len(team)

    cache = os.path.join(games.DATA_DIR, "cache", "atd_arrays_v1.npz")
    if use_cache and os.path.exists(cache) and os.path.getmtime(cache) >= os.path.getmtime(raw_path):
        z = np.load(cache)
        print("loaded from cache:", cache)
        return {k: z[k] for k in z.files}

    length, width = PITCH
    px = np.full((N_FRAMES, n_slots), np.nan, np.float32)
    py = np.full((N_FRAMES, n_slots), np.nan, np.float32)
    ps = np.full((N_FRAMES, n_slots), np.nan, np.float32)
    bx = np.full(N_FRAMES, np.nan, np.float32)
    by = np.full(N_FRAMES, np.nan, np.float32)

    started, n_bad = time.time(), 0
    with open(raw_path) as handle:
        for line in handle:
            parts = line.strip().split(":")
            if len(parts) != 3:
                n_bad += 1
                continue
            frame = int(parts[0]) - 1          # frame_id counts from 1, submission rows from 0
            if not 0 <= frame < N_FRAMES:
                n_bad += 1
                continue
            values = _parse_players(parts[1], n_slots)
            px[frame], py[frame], ps[frame] = values[:, 0], values[:, 1], values[:, 2]
            ball = parts[2].split(",")
            if len(ball) >= 2:
                try:
                    bx[frame], by[frame] = float(ball[0]), float(ball[1])
                except ValueError:
                    pass

    px *= length
    py *= width
    bx *= length
    by *= width
    arrays = dict(px=px, py=py, ps=ps, bx=bx, by=by, team=team)
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    np.savez(cache, **arrays)
    print("parsed {} in {:.0f} s, {} unreadable lines".format(
        os.path.basename(raw_path), time.time() - started, n_bad))
    return arrays


print("alignment:", load_atd.check_alignment_raw())
ARR = load_atd_arrays()

meta_path = games.require(["atd_meta"], GAME)[0]
HOME_NAME, AWAY_NAME = team_names(meta_path)
print("H = {}, A = {}".format(HOME_NAME, AWAY_NAME))
print("track slots: {} home, {} away".format(int((ARR["team"] == 0).sum()), int((ARR["team"] == 1).sum())))

### Hızlı kontrol
Bu sayılar mantıklı mı diye bak (top ~%57 karede görünür olmalı, oyun sırasında 20–22 oyuncu).

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    n_tracked = (~np.isnan(ARR["px"])).sum(axis=1)
    ball_seen = ~np.isnan(ARR["bx"])
    L, W = PITCH
    outside = ball_seen & ((ARR["bx"] < 0) | (ARR["bx"] > L) | (ARR["by"] < 0) | (ARR["by"] > W))
    print("ball visible            : {:.1f}% of frames".format(100 * ball_seen.mean()))
    print("ball outside the lines  : {:.2f}% of visible frames".format(
        100 * outside.sum() / max(1, ball_seen.sum())))
    print("players tracked (median): {:.0f}".format(np.median(n_tracked)))
    print("player speed field      : median {:.2f} m/s, 99th pct {:.2f} m/s".format(
        np.nanmedian(ARR["ps"]), np.nanpercentile(ARR["ps"], 99)))

## 2. Özellikler (features)

Her kare için: topun hızı, en yakın ev sahibi / deplasman oyuncusunun topa uzaklığı (`dH`, `dA`),
takip edilen oyuncu sayısı, oyuncuların ortalama hızı, top saha dışında mı.

In [ ]:
def runs(mask):
    """Half-open [start, stop) runs of True."""
    m = np.asarray(mask, dtype=np.int8)
    d = np.diff(np.concatenate(([0], m, [0])))
    return np.flatnonzero(d == 1), np.flatnonzero(d == -1)


def drop_short(mask, min_len):
    out = mask.copy()
    for a, b in zip(*runs(mask)):
        if b - a < min_len:
            out[a:b] = False
    return out


def close_gaps(mask, max_gap):
    """Fill False runs of at most max_gap frames that have True on both sides."""
    out = mask.copy()
    for a, b in zip(*runs(~mask)):
        if a > 0 and b < len(mask) and b - a <= max_gap:
            out[a:b] = True
    return out


def interp_short(x, max_gap):
    """Linear interpolation across NaN runs of at most max_gap frames, observed on both ends."""
    x = np.asarray(x, dtype=np.float64)
    good = ~np.isnan(x)
    if good.sum() < 2:
        return x.copy()
    idx = np.arange(len(x))
    filled = np.interp(idx, idx[good], x[good])
    out = x.copy()
    for a, b in zip(*runs(~good)):
        if a > 0 and b < len(x) and b - a <= max_gap:
            out[a:b] = filled[a:b]
    return out


def rolling_median(x, window):
    return pd.Series(x).rolling(window, center=True, min_periods=max(1, window // 4)).median().to_numpy()


def compute_features(arr, P):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        bx = interp_short(arr["bx"], P["BALL_INTERP_MAX"])
        by = interp_short(arr["by"], P["BALL_INTERP_MAX"])
        ball_raw_seen = ~np.isnan(arr["bx"])
        ball_seen = ~np.isnan(bx)

        step = np.hypot(np.diff(bx, prepend=np.nan), np.diff(by, prepend=np.nan)) * FPS
        bspeed = rolling_median(step, 5)

        d = np.hypot(arr["px"] - bx[:, None].astype(np.float32),
                     arr["py"] - by[:, None].astype(np.float32))
        d = np.where(np.isnan(d), np.inf, d)
        home = arr["team"] == 0
        dH = d[:, home].min(axis=1)
        dA = d[:, ~home].min(axis=1)

        n_tracked = (~np.isnan(arr["px"])).sum(axis=1)
        speed = np.clip(arr["ps"], 0, 12)
        if np.isnan(speed).all():   # no speed field: derive it from positions
            speed = np.hypot(np.diff(arr["px"], axis=0, prepend=np.nan),
                             np.diff(arr["py"], axis=0, prepend=np.nan)) * FPS
            speed = np.clip(speed, 0, 12)
        mean_speed = np.nanmean(speed, axis=1)
        mspeed = rolling_median(mean_speed, P["SPEED_WIN"])

        L, W, m = PITCH[0], PITCH[1], P["OUT_MARGIN"]
        rx, ry = arr["bx"], arr["by"]
        ball_out = ball_raw_seen & ((rx < -m) | (rx > L + m) | (ry < -m) | (ry > W + m))

    return dict(bx=bx, by=by, ball_raw_seen=ball_raw_seen, ball_seen=ball_seen, bspeed=bspeed,
                dH=dH, dA=dA, n_tracked=n_tracked, mspeed=mspeed, ball_out=ball_out)


F = compute_features(ARR, PARAMS)
print("features ready")


def activity_features(arr):
    """Player-movement features only (no ball), per frame, with past/future context.

    Follows the idea of predicting the in-game status from player tracking alone (Scientific
    Reports 2022, doi:10.1038/s41598-022-19948-1): speeds, accelerations and team shape, looked at
    over several seconds around each frame. No ball features, so the model cannot simply learn
    "ball invisible = dead", which is wrong 41% of the time in this data.
    """
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        px, py = arr["px"], arr["py"]
        sp = np.clip(arr["ps"], 0, 12)
        if np.isnan(sp).all():
            sp = np.clip(np.hypot(np.diff(px, axis=0, prepend=np.nan),
                                  np.diff(py, axis=0, prepend=np.nan)) * FPS, 0, 12)
        tracked = ~np.isnan(px)
        n_tr = np.maximum(tracked.sum(axis=1), 1)
        filled = np.where(np.isnan(sp), -1.0, sp)
        top3 = np.sort(filled, axis=1)[:, -3:]
        top3 = np.where(top3 < 0, np.nan, top3)
        acc = np.abs(np.diff(sp, axis=0, prepend=np.nan)) * FPS
        base = {
            "mean_speed": np.nanmean(sp, axis=1),
            "top3_speed": np.nanmean(top3, axis=1),
            "frac_over_2": (sp > 2).sum(axis=1) / n_tr,
            "frac_over_4": (sp > 4).sum(axis=1) / n_tr,
            "mean_acc": np.clip(np.nanmean(acc, axis=1), 0, 10),
            "n_tracked": np.minimum(n_tr, 24),
        }
        cen = {}
        for side, mask in ((0, arr["team"] == 0), (1, arr["team"] == 1)):
            cx, cy = np.nanmean(px[:, mask], axis=1), np.nanmean(py[:, mask], axis=1)
            cen[side] = (cx, cy)
            v = np.hypot(np.diff(cx, prepend=np.nan), np.diff(cy, prepend=np.nan)) * FPS
            base["centroid_speed_{}".format(side)] = np.clip(rolling_median(v, 5), 0, 8)
            base["spread_{}".format(side)] = np.nanstd(px[:, mask], axis=1) + np.nanstd(py[:, mask], axis=1)
        base["centroid_gap"] = np.hypot(cen[0][0] - cen[1][0], cen[0][1] - cen[1][1])

    frame = pd.DataFrame(base)
    parts = []
    for seconds in (1, 4, 10):
        w = seconds * FPS
        parts.append(frame.rolling(w, center=True, min_periods=1).mean().add_suffix("_c{}".format(seconds)))
    key = frame[["mean_speed", "top3_speed", "frac_over_2", "mean_acc"]]
    four = key.rolling(4 * FPS, min_periods=1).mean()
    parts.append(four.shift(2 * FPS).add_suffix("_past4"))      # the 4 s before
    parts.append(four.shift(-6 * FPS).add_suffix("_next4"))     # the 4 s after
    X = pd.concat(parts, axis=1)
    X = X.fillna(X.median())
    return X.to_numpy(np.float32), list(X.columns)


X_ACT, X_ACT_NAMES = activity_features(ARR)
print("activity features:", X_ACT.shape)

## 3. Smart Tagging: duran toplar ve oyun fazları

`Team` değerleri kulüp adı (`Barcelona`, `Atletic Club`, `N/A`); bunları H/A'ya isim
benzerliğiyle çeviriyoruz. Taçlarda takım `N/A` olduğu için o durumda tracking karar veriyor.

In [ ]:
def side_of_tag(value, home_name, away_name):
    """'H', 'A' or None for a Smart Tagging Team value, by name similarity."""
    if value is None:
        return None
    v = str(value).strip().lower()
    if v in ("", "n/a", "na", "none"):
        return None
    sim_home = difflib.SequenceMatcher(None, v, str(home_name).lower()).ratio()
    sim_away = difflib.SequenceMatcher(None, v, str(away_name).lower()).ratio()
    if max(sim_home, sim_away) < 0.4 or abs(sim_home - sim_away) < 0.05:
        return None
    return "H" if sim_home > sim_away else "A"


TAGS = load_smart_tagging.load()
print("time base check:", load_smart_tagging.check_time_base(TAGS)["agrees"])
for value in sorted({t["tags"].get("Team") for t in TAGS} - {None}):
    print("  Team tag {!r:>16} -> {}".format(value, side_of_tag(value, HOME_NAME, AWAY_NAME)))

SET_PIECES = [t for t in TAGS if (t["code"] or "").strip().lower() == "set pieces"]
print("{} set pieces, {} phase tags".format(len(SET_PIECES), len(TAGS) - len(SET_PIECES)))

## 4. Ölü top (`D`) tespiti

Bir kare `D` sayılır, eğer:
1. Sahada 12'den az oyuncu takip ediliyorsa (devre arası, maç öncesi),
2. Top saha dışına çıktıysa → top tekrar sahada hızla oynanana kadar,
3. Top 2 sn boyunca yerde hareketsiz duruyorsa ve kimse dibinde değilse (faul sonrası serbest vuruş için konmuş top),
4. Smart Tagging'de bir duran top varsa: etiketin başından gerçek vuruşa kadar,
5. **Öğrenilen model** öyle diyorsa (aşağıda).

**Öğrenilen model (self-supervised):** Top bize bazı kareler için kesin bilgi veriyor: top saha dışında,
yerde duruyor ya da duran top öncesi → *ölü*; top bir oyuncunun kontrolünde ya da hızlı pas halinde → *canlı*.
Bu kareler üzerinde, **sadece oyuncu hareketlerinden** (hız, ivme, takım yayılımı, ±4 sn geçmiş/gelecek)
küçük bir lojistik regresyon eğitiliyor ve maçın geri kalanına uygulanıyor. Fikir: Scientific Reports 2022,
"in-game status from player tracking" (doi:10.1038/s41598-022-19948-1). Hiçbir insan etiketi kullanılmıyor
(kurallar algoritmik pseudo-label'a izin veriyor).

**Güvenlik ağı:** Model, bir devreden öğrenip diğer devreyi %70'ten kötü tahmin ederse ya da ölü top oranı
0,20–0,65 dışında çıkarsa (yayınlanmış maçlarda ~0,30–0,53), otomatik olarak eski Otsu kuralına dönülüyor.

Sonra temizlik: 1,5 sn'den kısa "canlı" boşluklar dolduruluyor, 3 sn'den kısa `D` parçaları siliniyor.

In [ ]:
def otsu_threshold(values, lo=0.0, hi=6.0, bins=120):
    v = values[np.isfinite(values)]
    v = v[(v >= lo) & (v <= hi)]
    hist, edges = np.histogram(v, bins=bins, range=(lo, hi))
    centers = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(hist).astype(np.float64)
    w1 = w0[-1] - w0
    m0 = np.cumsum(hist * centers)
    with np.errstate(divide="ignore", invalid="ignore"):
        between = w0 * w1 * (m0 / w0 - (m0[-1] - m0) / w1) ** 2
    between[~np.isfinite(between)] = -1
    return float(edges[np.argmax(between) + 1])


def fit_logistic(X, y, weight, l2=1.0, iters=30):
    """Weighted, L2-regularised logistic regression by Newton's method. Deterministic."""
    Xb = np.c_[np.ones(len(X)), X].astype(np.float64)
    beta = np.zeros(Xb.shape[1])
    ridge = l2 * np.r_[0.0, np.ones(Xb.shape[1] - 1)]
    for _ in range(iters):
        p = 1.0 / (1.0 + np.exp(-np.clip(Xb @ beta, -30, 30)))
        grad = Xb.T @ (weight * (p - y)) + ridge * beta
        hess = (Xb * (weight * p * (1 - p))[:, None]).T @ Xb + np.diag(ridge)
        step = np.linalg.solve(hess, grad)
        beta -= step
        if np.max(np.abs(step)) < 1e-6:
            break
    return beta


def predict_logistic(beta, X):
    return 1.0 / (1.0 + np.exp(-np.clip(np.c_[np.ones(len(X)), X] @ beta, -30, 30)))


def learn_stoppage_probability(X, dead_seed, alive_seed, in_play, report=None):
    """Self-supervised stoppage model: train on frames the ball tells us about for certain,
    predict everywhere from player movement. Pseudo-labels come from algorithms only."""
    d_idx = np.flatnonzero(dead_seed & in_play)
    a_idx = np.flatnonzero(alive_seed & ~dead_seed & in_play)
    if len(d_idx) < 500 or len(a_idx) < 500:
        return None
    step_d, step_a = max(1, len(d_idx) // 40000), max(1, len(a_idx) // 40000)
    idx = np.r_[d_idx[::step_d], a_idx[::step_a]]
    y = np.r_[np.ones(len(d_idx[::step_d])), np.zeros(len(a_idx[::step_a]))]
    w = np.where(y == 1, 0.5 / (y == 1).sum(), 0.5 / (y == 0).sum()) * len(y)
    mu, sd = X[idx].mean(axis=0), X[idx].std(axis=0) + 1e-6
    Z = (X - mu) / sd

    if report is not None:   # honesty check: train on one half, test on the other (seeds only)
        half = np.zeros(len(X), bool)
        half[HALF_BLOCKS[0][0]:HALF_BLOCKS[0][1]] = True
        scores = []
        for train_first in (True, False):
            tr = half[idx] if train_first else ~half[idx]
            beta = fit_logistic(Z[idx[tr]], y[tr], w[tr])
            pred = predict_logistic(beta, Z[idx[~tr]]) > 0.5
            truth = y[~tr] == 1
            scores.append(0.5 * (pred[truth].mean() + (~pred[~truth]).mean()))
        report["seed_dead"], report["seed_alive"] = int(len(d_idx)), int(len(a_idx))
        report["cross_half_balanced_acc"] = round(float(np.mean(scores)), 3)
        report["seed_dead_share"] = round(len(d_idx) / float(len(d_idx) + len(a_idx)), 3)

    beta = fit_logistic(Z[idx], y, w)
    return predict_logistic(beta, Z)


def detect_dead(F, P, set_pieces, events=None, X_act=None, report=None):
    n = len(F["mspeed"])
    few = F["n_tracked"] < P["MIN_PLAYERS"]
    in_play = np.zeros(n, bool)
    for a, b in HALF_BLOCKS:
        in_play[a:b] = True
    with np.errstate(invalid="ignore"):
        restart_candidate = (F["ball_raw_seen"] & ~F["ball_out"]
                             & (F["bspeed"] > P["RESTART_SPEED"]))
    candidates = np.flatnonzero(restart_candidate)

    # 1. ball out of play -> dead until the ball is next played inside the pitch
    out_hold = np.zeros(n, bool)
    out_events = drop_short(close_gaps(F["ball_out"], 2), 3)
    for a, b in zip(*runs(out_events)):
        j = np.searchsorted(candidates, b)
        stop = candidates[j] if j < len(candidates) else n
        out_hold[a:min(stop, a + P["MAX_OUT_HOLD"])] = True

    # 2. ball placed for a free kick: visible, still for 2 s, nobody on it
    static = np.zeros(n, bool)
    if P["USE_STATIC_BALL"]:
        win = P["STATIC_WIN"]
        roll_x = pd.Series(F["bx"]).rolling(win, center=True, min_periods=int(0.8 * win))
        roll_y = pd.Series(F["by"]).rolling(win, center=True, min_periods=int(0.8 * win))
        span = np.hypot((roll_x.max() - roll_x.min()).to_numpy(), (roll_y.max() - roll_y.min()).to_numpy())
        nearest = rolling_median(np.minimum(F["dH"], F["dA"]), FPS)
        with np.errstate(invalid="ignore"):
            static = (span < P["STATIC_SPAN"]) & (nearest > P["STATIC_CLEAR"])

    # 3. set pieces: dead from the tag's start to the kick, alive for 2 s after it
    restarts = []
    set_piece_dead = np.zeros(n, bool)
    forced_alive = np.zeros(n, bool)
    if P["USE_SET_PIECES"]:
        lead = int(round(P["SET_PIECE_LEAD"] * FPS))
        for tag in set_pieces:
            s, e = max(0, tag["start_frame"]), min(n, tag["end_frame"])
            j = np.searchsorted(candidates, s + FPS // 2)
            kick = candidates[j] if j < len(candidates) and candidates[j] < e + 2 * FPS else s + lead
            kick = int(min(kick, n - 1))
            set_piece_dead[s:kick] = True
            forced_alive[kick:kick + 2 * FPS] = True
            restarts.append((kick, side_of_tag(tag["tags"].get("Team"), HOME_NAME, AWAY_NAME)))
    restarts.sort(key=lambda r: r[0])

    # 4. everything else: how the players move
    thr = float("nan")
    activity_dead = np.zeros(n, bool)
    prob = None
    if P["DEAD_MODEL"] == "learned" and X_act is not None and events is not None:
        controlled = np.zeros(n, bool)
        for a, b, _ in events:
            controlled[a:b] = True
        with np.errstate(invalid="ignore"):
            passing = F["ball_raw_seen"] & ~F["ball_out"] & (F["bspeed"] > 8.0)
        dead_seed = out_hold | static | set_piece_dead
        alive_seed = (controlled | passing) & ~forced_alive & ~close_gaps(dead_seed, 2 * FPS)
        prob = learn_stoppage_probability(X_act, dead_seed, alive_seed, in_play & ~few, report)
        if prob is not None:
            smooth = rolling_median(prob, P["MODEL_SMOOTH"])
            activity_dead = drop_short(close_gaps(smooth > 0.5, int(1.5 * FPS)), P["MODEL_MIN_DEAD"])
    if prob is None and P["USE_SLOW_RULE"]:   # v3 behaviour, also the fallback
        basis = ~few & (in_play if P["OTSU_IN_HALVES"] else True)
        thr = float(np.clip(otsu_threshold(F["mspeed"][basis]), P["SLOW_MIN"], P["SLOW_MAX"]))
        with np.errstate(invalid="ignore"):
            slow = F["mspeed"] < thr
            quiet_speed = rolling_median(F["bspeed"], FPS)
            ball_quiet = np.isnan(quiet_speed) | (quiet_speed < P["BALL_QUIET_SPEED"])
        activity_dead = slow & ball_quiet
    if report is not None:
        report["dead_model"] = "learned" if prob is not None else "otsu"
        report["slow_thr"] = thr

    dead = few | activity_dead | out_hold | static
    dead = close_gaps(dead, P["MAX_ALIVE_GAP_IN_DEAD"])
    dead = drop_short(dead, P["MIN_DEAD"])
    dead = (dead | set_piece_dead) & ~forced_alive

    # safety net: an implausible learned result falls back to the v3 rule (no ground truth used)
    if prob is not None:
        share = float(dead[in_play].mean())
        lo, hi = P["PLAUSIBLE_DEAD"]
        acc = report.get("cross_half_balanced_acc", 1.0) if report is not None else 1.0
        if not (lo <= share <= hi) or acc < P["MIN_MODEL_ACC"]:
            if report is not None:
                report["fallback_reason"] = "dead share {:.2f}, cross-half accuracy {}".format(share, acc)
            return detect_dead(F, {**P, "DEAD_MODEL": "otsu"}, set_pieces, events, X_act, report)
    return dead, restarts, thr


# (run inside run_pipeline below, because the stoppage model also uses the control events)

## 5. Kim kontrol ediyor? → Possession

**Kontrol olayı:** top görünür, yavaş (< 12 m/s) ve bir takımın oyuncusu ≤ 2 m mesafede ve
rakipten en az 0,75 m daha yakın; bu durum en az 6 kare (0,24 sn) sürüyor.
Tek dokunuş, sekme, havada kazanılan ama kontrol edilmeyen top bu kadar uzun sürmez → possession değişmez.

**Durum makinesi** (her canlı aralık için):
- Başlangıç takımı: yakın bir duran top etiketinin takımı → yoksa ilk kontrol olayının takımı → yoksa faz etiketleri → yoksa önceki takım.
- Rakip takımın kontrol olayı possession'ı **ancak** şu durumda değiştirir: topu ≥ 1 sn tutuyor **ya da**
  bir sonraki kontrol olayı da onun takımının (challenge kuralı: "kontrolsüz dokunuş, sekme, sonrasında kontrol
  edilmeyen hava topu possession değiştirmez").
- Top bir süre görünmezken el değiştirdiyse, değişim anı boşluğun ortasına konur (beklenen hatayı en aza indirir).
- Top kaybolsa bile possession son takımda kalır.

**Faz etiketleri** (BUILD UP, DEFENDING IN ... vs.): Her kodun `Team` değeri topa sahip olan takımı mı,
savunan takımı mı gösteriyor? Bunu ground truth'a bakmadan, **kendi kontrol olaylarımızla uyumuna**
bakarak otomatik çıkarıyoruz. Sadece uyumu net olan kodlar (≥ 0,8 veya ≤ 0,2) kullanılıyor ve
sadece tracking'in kontrol görmediği yerlerde devreye giriyor.

In [ ]:
def control_events(F, P):
    """[(start, stop, side)] with side 0 = home, 1 = away, sorted by start."""
    with np.errstate(invalid="ignore"):
        ok = F["ball_seen"] & ~(F["bspeed"] > P["BALL_MAX_SPEED_CTRL"])
    R, M = P["R_CTRL"], P["MARGIN"]
    dH, dA = F["dH"], F["dA"]
    masks = {0: ok & (dH <= R) & (dH + M <= dA), 1: ok & (dA <= R) & (dA + M <= dH)}
    events = []
    for side, mask in masks.items():
        mask = drop_short(close_gaps(mask, P["K_GAP"]), P["K_CTRL"])
        events += [(int(a), int(b), side) for a, b in zip(*runs(mask))]
    events.sort()
    return events


def phase_prior(tags, events, n, min_agreement=0.8):
    """Per-frame vote, +1 home / -1 away, from phase tags whose meaning is learned from our own
    control events (never from ground truth). Returns (prior, {code: sign})."""
    ctrl = np.zeros(n, np.int8)                 # +1 home control, -1 away control, 0 none
    for a, b, side in events:
        ctrl[a:b] = 1 if side == 0 else -1
    by_code = {}
    for tag in tags:
        code = (tag["code"] or "").strip().upper()
        if code in ("SET PIECES", "GOALS"):
            continue
        side = side_of_tag(tag["tags"].get("Team"), HOME_NAME, AWAY_NAME)
        if side is None:
            continue
        by_code.setdefault(code, []).append((max(0, tag["start_frame"]), min(n, tag["end_frame"]),
                                             1 if side == "H" else -1))
    prior, signs = np.zeros(n, np.float32), {}
    for code, spans in by_code.items():
        agree = total = 0
        for a, b, s in spans:
            c = ctrl[a:b]
            c = c[c != 0]
            agree += int((c == s).sum())
            total += len(c)
        if total < 200:
            continue
        rate = agree / total
        sign = 1 if rate >= min_agreement else (-1 if rate <= 1 - min_agreement else 0)
        signs[code] = (sign, round(rate, 2), total)
        if sign:
            for a, b, s in spans:
                prior[a:b] += sign * s
    return prior, signs


def assign_possession(dead, events, restarts, prior, P, last_seen=None):
    n = len(dead)
    labels = np.full(n, "D", dtype="<U1")
    starts = np.array([e[0] for e in events], dtype=np.int64)
    r_frames = np.array([r[0] for r in restarts], dtype=np.int64)
    last = 0
    for a, b in zip(*runs(~dead)):
        seg = events[np.searchsorted(starts, a):np.searchsorted(starts, b)]

        side = None
        k = np.searchsorted(r_frames, a - P["RESTART_TOL"])
        while side is None and k < len(r_frames) and r_frames[k] <= a + P["RESTART_TOL"]:
            if restarts[k][1] is not None:
                side = 0 if restarts[k][1] == "H" else 1
            k += 1
        if side is None:
            head_stop = seg[0][0] if seg else b
            vote = prior[a:head_stop].sum() if prior is not None else 0
            if seg and (seg[0][0] - a <= 3 * FPS or vote == 0):
                side = seg[0][2]
            elif vote != 0:
                side = 0 if vote > 0 else 1
            else:
                side = last

        codes = np.empty(b - a, np.int8)
        cur, pos = side, a
        last_own_end = a                      # end of the current team's latest control
        for i, (s, e, t) in enumerate(seg):
            if t == cur:
                last_own_end = max(last_own_end, e)
                continue
            if P["REQUIRE_CONFIRM"]:
                kept = e - s >= P["KEEP"]
                followed = i + 1 < len(seg) and seg[i + 1][2] == t
                if not (kept or followed):
                    continue                  # a touch, a block, a deflection: not control
            switch = s
            if P["GAP_MIDPOINT"] and last_seen is not None and s > a:
                gap_start = max(last_seen[s - 1] + 1, last_own_end, pos)
                if s - gap_start >= FPS // 2:  # the ball was hidden for a while before this
                    switch = (gap_start + s) // 2
            codes[pos - a:switch - a] = cur
            pos, cur = switch, t
            last_own_end = e
        codes[pos - a:] = cur
        labels[a:b] = np.where(codes == 0, "H", "A")
        last = cur
    return labels


def remove_short_spells(labels, min_len, rounds=5):
    """Merge H/A runs shorter than min_len into their neighbours; isolated blips become D."""
    lab = labels.copy()
    alive = ("H", "A")
    for _ in range(rounds):
        cut = np.flatnonzero(lab[1:] != lab[:-1]) + 1
        st, en = np.r_[0, cut], np.r_[cut, len(lab)]
        vals = lab[st]
        changed = False
        for i in range(len(st)):
            if vals[i] == "D" or en[i] - st[i] >= min_len:
                continue
            left = vals[i - 1] if i > 0 else None
            right = vals[i + 1] if i + 1 < len(st) else None
            if left in alive and right in alive:
                if left != right:
                    continue            # a genuine short possession between two others
                new = left
            elif left in alive:
                new = left
            elif right in alive:
                new = right
            else:
                new = "D"
            lab[st[i]:en[i]] = new
            changed = True
        if not changed:
            break
    return lab


def run_pipeline(F, P, tags, set_pieces, verbose=True, report=None):
    report = {} if report is None else report
    events = control_events(F, P)
    dead, restarts, thr = detect_dead(F, P, set_pieces, events, X_ACT, report)
    prior, signs = (phase_prior(tags, events, len(dead), P["PRIOR_MIN_AGREEMENT"])
                    if P["USE_PHASE_PRIOR"] else (None, {}))
    idx = np.arange(len(dead))
    last_seen = np.maximum.accumulate(np.where(F["ball_seen"], idx, -1))
    labels = assign_possession(dead, events, restarts, prior, P, last_seen)
    labels = remove_short_spells(labels, P["MIN_SPELL"])
    if verbose:
        print("control events: {}  |  restarts from tags: {}  |  stoppage model: {}".format(
            len(events), len(restarts), report.get("dead_model")))
        if report.get("dead_model") == "learned":
            print("  pseudo-labels: {} dead / {} alive frames; cross-half balanced accuracy {}".format(
                report["seed_dead"], report["seed_alive"], report["cross_half_balanced_acc"]))
        else:
            print("  slow-play threshold (Otsu): {:.2f} m/s".format(thr))
        if report.get("fallback_reason"):
            print("  NOTE: learned model rejected ({}), used the Otsu rule".format(report["fallback_reason"]))
        print("  dead frames: {:.1f}% of the video".format(100 * np.mean(labels == "D")))
        for code, (sign, rate, total) in sorted(signs.items()):
            meaning = {1: "Team = in possession", -1: "Team = defending", 0: "unclear, unused"}[sign]
            print("  phase {:<30} agreement {:.2f} on {:>6} frames -> {}".format(code, rate, total, meaning))
    return labels


LABELS = run_pipeline(F, PARAMS, TAGS, SET_PIECES)

## 6. Tüm maç için yapısal kontrol

Ground truth'a bakmadan: kaç possession, ortalama süre, `D` oranı. Gerçek bir maçta
**birkaç yüz possession, ortalama ~10–20 sn** beklenir. Binlerce possession = titreme var demektir.
(Oyunun oynandığı aralıklar `games.py`'deki yayınlanmış devre sınırlarından alınıyor.)

In [ ]:
def structure(labels):
    rows = {"possessions": 0, "alive_frames": 0, "D_frames": 0, "H_frames": 0, "frames": 0}
    stoppage_lengths = []
    for a, b in HALF_BLOCKS:
        lab = labels[a:b]
        rows["frames"] += len(lab)
        rows["D_frames"] += int((lab == "D").sum())
        rows["H_frames"] += int((lab == "H").sum())
        alive = np.isin(lab, ["H", "A"])
        rows["alive_frames"] += int(alive.sum())
        cut = np.flatnonzero(lab[1:] != lab[:-1]) + 1
        rows["possessions"] += int(np.isin(lab[np.r_[0, cut]], ["H", "A"]).sum())
        stoppage_lengths += [(b2 - a2) / FPS for a2, b2 in zip(*runs(lab == "D"))]
    return {
        "possessions": rows["possessions"],
        "mean_possession_s": round(rows["alive_frames"] / max(1, rows["possessions"]) / FPS, 1),
        "dead_share": round(rows["D_frames"] / rows["frames"], 3),
        "stoppages": len(stoppage_lengths),
        "median_stoppage_s": round(float(np.median(stoppage_lengths)), 1) if stoppage_lengths else 0.0,
        "home_share_alive": round(rows["H_frames"] / max(1, rows["alive_frames"]), 3),
    }


def read_labels(path):
    return pm.read_label_csv(path)["label"].astype(str).str.strip().str.upper().to_numpy()


BASELINE50 = read_labels(os.path.join(ROOT, "evaluation", "results", "baseline_smooth50.csv"))
print(pd.DataFrame({"ours": structure(LABELS), "baseline_smooth50": structure(BASELINE50)}))

## 7. Public 5 dakika: pipeline doğru mu?

⚠️ Sadece 17 possession var. Buradaki skorla parametre seçme — `EVALUATION.md` bunun yanlış
sonuca götürdüğünü gösteriyor. Amaç: etiketler hizalı mı, H/A ters mi, `D` mantıklı mı?
Referans: `baseline_smooth50` public pencerede spellF1 **0.3256**, tüm maçta **0.2433**.

In [ ]:
GT_PUBLIC = os.path.join(ROOT, "evaluation", "gt", "BAR-ATH_possession_gt_public.csv")
GT_CODES = pm.label_codes(pm.read_label_csv(GT_PUBLIC))

KEYS = ["spell_f1_mean", "macro_f1", "n_pred_spells", "n_gt_spells", "alive_dead_f1",
        "team_acc_gt_alive", "mean_spell_duration_s", "gt_mean_spell_duration_s"]


def public_scores(labels):
    result = pm.evaluate(GT_CODES, pm.label_codes(labels))
    return {k: (round(result[k], 4) if isinstance(result[k], float) else result[k]) for k in KEYS}


print(pd.DataFrame({"ours": public_scores(LABELS), "baseline_smooth50": public_scores(BASELINE50)}))

### Görsel kontrol (public pencere)
Üstte gerçek etiketler, ortada bizimki, altta baseline. Mavi = H, turuncu = A, gri = D.

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap

    a, b = 11185, 18685
    to_num = {"H": 0, "A": 1, "D": 2, "X": 3}
    gt_labels = np.array(pm.LABELS)[GT_CODES]
    rows = [gt_labels[a:b], LABELS[a:b], BASELINE50[a:b]]
    grid = np.array([[to_num[v] for v in row] for row in rows])
    cmap = ListedColormap(["#1f77b4", "#ff7f0e", "#9e9e9e", "#ffffff"])
    fig, ax = plt.subplots(figsize=(14, 2.2))
    ax.imshow(grid, aspect="auto", cmap=cmap, vmin=0, vmax=3, interpolation="nearest",
              extent=(0, (b - a) / FPS, 3, 0))
    ax.set_yticks([0.5, 1.5, 2.5])
    ax.set_yticklabels(["ground truth", "ours", "baseline50"])
    ax.set_xlabel("seconds from the first-half kickoff")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed; skipping the plot")

## 8. Ablation: her parça ne katıyor?

İlk satır gönderilen ayar (v4). Diğer satırlar bir parçayı kapatınca ne olduğunu gösterir; teknik
dokümanda kullanılacak. Karar verirken **tüm maç yapısına** (ölü top oranı, durma sayısı, possession süresi)
bak, public skora değil.

In [ ]:
VARIANTS = {
    "v4 (submitted config)": {},
    "v3 complete (old)": {"DEAD_MODEL": "otsu", "REQUIRE_CONFIRM": False, "GAP_MIDPOINT": False,
                          "USE_STATIC_BALL": False},
    "dead ball: Otsu rule instead of model": {"DEAD_MODEL": "otsu"},
    "no confirmation of gains": {"REQUIRE_CONFIRM": False},
    "no gap midpoint": {"GAP_MIDPOINT": False},
    "no static-ball rule": {"USE_STATIC_BALL": False},
    "no set pieces": {"USE_SET_PIECES": False},
    "no phase prior": {"USE_PHASE_PRIOR": False},
    "no short-spell cleanup": {"MIN_SPELL": 1},
    "stricter control (K=10)": {"K_CTRL": 10},
    "looser control (R=2.5 m)": {"R_CTRL": 2.5},
}
rows = {}
for name, change in VARIANTS.items():
    info = {}
    labels = run_pipeline(F, {**PARAMS, **change}, TAGS, SET_PIECES, verbose=False, report=info)
    pub = public_scores(labels)
    rows[name] = {**structure(labels), "model": info.get("dead_model"),
                  "public_spellF1": pub["spell_f1_mean"], "public_macroF1": pub["macro_f1"]}
ABLATION = pd.DataFrame(rows).T
print(ABLATION.to_string())

## 9. Submission dosyasını yaz ve doğrula

Dosya `evaluation/results/<takım_adı>.csv` olarak yazılır (bu klasör `.gitignore`'da, GitHub'a gitmez).
Resmi validator ile kontrol edilir. Forma **bu dosyayı** yükle.

In [ ]:
def write_submission(labels, team_name):
    path = os.path.join(ROOT, "evaluation", "results", "{}.csv".format(team_name))
    pd.DataFrame({"frame": np.arange(len(labels)), "label": labels}).to_csv(
        path, index=False, lineterminator="\n")
    return path


assert len(LABELS) == N_FRAMES and set(np.unique(LABELS)) <= {"H", "A", "D", "X"}
SUB_PATH = write_submission(LABELS, TEAM_NAME)
print("written:", SUB_PATH)

check = subprocess.run([sys.executable, os.path.join(ROOT, "evaluation", "validate_submission.py"), SUB_PATH],
                       capture_output=True, text=True)
print(check.stdout or check.stderr)

In [ ]:
# Optional: the official scorer over every CSV in evaluation/results/ (public 5 minutes).
score = subprocess.run([sys.executable, os.path.join(ROOT, "evaluation", "run_evaluation.py"), "--no-detail"],
                       capture_output=True, text=True, cwd=ROOT)
print(score.stdout[-3000:] or score.stderr[-3000:])

## 10. Teslimden önce

1. **Repo:** Bu notebook + `requirements.txt` (numpy, pandas, matplotlib sürümlerini sabitle: `pip freeze | grep -E "numpy|pandas|matplotlib"`).
   Yeniden üretme komutu: `jupyter nbconvert --to notebook --execute possession_pipeline.ipynb --output executed.ipynb`
   Rastgelelik yok, aynı veriyle **aynı dosya** çıkar (kural 3).
2. **Teknik doküman** (PDF veya .md, birkaç sayfa): yaklaşım, kullanılan veriler (ATD, Smart Tagging; video ve homography kullanılmadı),
   possession kuralları (bölüm 5), ölü top kuralları (bölüm 4), sınırlamalar, nasıl çalıştırılır. Bölüm 8'deki ablation tablosu burada işe yarar.
3. **Form:** takım adı (kayıttakiyle aynı), `<takım_adı>.csv`, repo linki, doküman. Son gün **17 Kasım**.

**Geliştirme fikirleri (vakit kalırsa):**
- Topun %41 kayıp olduğu yerlerde: oyuncuların hareket yönü / baskı yapısıyla possession tahmini.
- Kontrol olayları için sabit eşik yerine küçük bir sınıflandırıcı (ama etiket olarak sadece algoritmik pseudo-label!).
- Broadcast kamerası veya videodan kendi top tespitin (zor, ama kurallara uygun).